# Build Verified GSI Files — `fixed_batches_20_GSI_all_entities`

For every species folder across all 20 batches:
1. Find the digit-prefixed file ending with `_gsi_audit.json` (e.g. `0001_Abatus_cordatus_abj_gsi_audit.json`)
2. Extract only the four fields: `id`, `specie`, `specie_en`, `corrected_text_final`
3. Write a new file in the **same species folder**, renaming `_gsi_audit.json` → `_verified.json`  
   e.g. `0001_Abatus_cordatus_abj_verified.json`

In [1]:
import json
from pathlib import Path

base_dir = Path("../fixed_batches_20_GSI_all_entities")

KEEP_FIELDS = ["id", "specie", "specie_en", "corrected_text_final"]

created   = 0
skipped   = 0
no_field  = []

for batch_dir in sorted(base_dir.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue
    for species_dir in sorted(batch_dir.iterdir()):
        if not species_dir.is_dir():
            continue
        for f in species_dir.iterdir():
            if not (f.name[0].isdigit() and f.name.endswith("_gsi_audit.json")):
                continue

            dest_name = f.name.replace("_gsi_audit.json", "_verified.json")
            dest_path = species_dir / dest_name

            if dest_path.exists():
                skipped += 1
                continue

            data = json.loads(f.read_text())

            if "corrected_text_final" not in data:
                no_field.append(str(f))
                continue

            slim = {k: data[k] for k in KEEP_FIELDS if k in data}
            dest_path.write_text(json.dumps(slim, indent=2, ensure_ascii=False))
            created += 1

print(f"Files created : {created}")
print(f"Files skipped (already exist): {skipped}")
if no_field:
    print(f"\nWarning — {len(no_field)} files had no 'corrected_text_final' field:")
    for p in no_field:
        print(" ", p)
else:
    print("All source files had 'corrected_text_final'.")

Files created : 7281
Files skipped (already exist): 0
All source files had 'corrected_text_final'.


In [2]:
# Quick sanity check — show one example output file
example = next(
    f for batch_dir in sorted(base_dir.iterdir())
    if batch_dir.is_dir() and batch_dir.name.startswith("batch_")
    for species_dir in sorted(batch_dir.iterdir())
    if species_dir.is_dir()
    for f in species_dir.iterdir()
    if f.name.endswith("_verified.json")
)
print(f"Example: {example}\n")
data = json.loads(example.read_text())
print("Keys:", list(data.keys()))
print("id     :", data.get("id"))
print("specie :", data.get("specie"))
print("specie_en:", data.get("specie_en"))
print("corrected_text_final (first 200 chars):", data.get("corrected_text_final", "")[:200])

Example: ../fixed_batches_20_GSI_all_entities/batch_01/Abatus_cordatus/0001_Abatus_cordatus_abj_verified.json

Keys: ['id', 'specie', 'specie_en', 'corrected_text_final']
id     : 1
specie : Abatus_cordatus
specie_en: Kerguelen sea urchin
corrected_text_final (first 200 chars): Field surveys at Anse du Halage have provided extensive data on the life history of the Kerguelen sea urchin Abatus cordatus. Among the most notable findings is the maximum observed length recorded fo


In [3]:
# Rename "corrected_text_final" → "fixed_text" in all _verified.json files
renamed  = 0
skipped  = 0

for batch_dir in sorted(base_dir.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue
    for species_dir in sorted(batch_dir.iterdir()):
        if not species_dir.is_dir():
            continue
        for f in species_dir.iterdir():
            if not (f.name[0].isdigit() and f.name.endswith("_verified.json")):
                continue
            data = json.loads(f.read_text())
            if "corrected_text_final" not in data:
                skipped += 1
                continue
            data["fixed_text"] = data.pop("corrected_text_final")
            f.write_text(json.dumps(data, indent=2, ensure_ascii=False))
            renamed += 1

print(f"Files updated (key renamed): {renamed}")
print(f"Files skipped (key already absent): {skipped}")

# Sanity check — show keys of the first verified file
example = next(
    f for batch_dir in sorted(base_dir.iterdir())
    if batch_dir.is_dir() and batch_dir.name.startswith("batch_")
    for species_dir in sorted(batch_dir.iterdir())
    if species_dir.is_dir()
    for f in species_dir.iterdir()
    if f.name.endswith("_verified.json")
)
print(f"\nSample file: {example.name}")
print("Keys after rename:", list(json.loads(example.read_text()).keys()))

Files updated (key renamed): 7281
Files skipped (key already absent): 0

Sample file: 0001_Abatus_cordatus_abj_verified.json
Keys after rename: ['id', 'specie', 'specie_en', 'fixed_text']
